# Sources from connectors

From a warehouse table to a causal model without leaving Python: register a
connector, author a custom SQL query against the live database, import the
result as a source, and model it.

This transcript uses a PostgreSQL database; Snowflake, MySQL, ClickHouse,
MongoDB, S3, and the other connector types follow the same verbs — swap the
`type` and credentials. Credentials are stored encrypted server-side and are
never returned by the API.

In [ ]:
import os

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))
ws = rc.workspace("connector-demo", create=True)

## Register and test

One call to register, one to prove the credentials reach the database:

In [ ]:
connector = ws.add_connector(
    "demo-warehouse", "PostgreSQL",
    host=os.environ.get("DEMO_DB_HOST", "localhost"), port=5455,
    database="warehouse", username="demo", password=os.environ.get("DEMO_DB_PASSWORD", "demo"),
)
connector.test()

## Browse the schema

The same hierarchy the UI shows — schemas, then tables, then columns:

In [ ]:
connector.browse("tables", schema="public")

## Author custom SQL against the live database

`query()` runs your SQL with a row cap and returns sample rows — nothing is
stored, and database errors come back verbatim, so the authoring loop stays
tight:

In [ ]:
try:
    connector.query("SELECT * FROM store_week")
except rc.RootCauseError as error:
    print(error)

In [ ]:
connector.query(
    "SELECT region, week, marketing_spend, conversions, revenue FROM store_weeks ORDER BY week",
    limit=5,
)

## Import the query result as a source

The same SQL, minus the safety net: `import_query()` materialises the full
result set as a source in the workspace and blocks until ingest completes.

In [ ]:
source = connector.import_query(
    "SELECT region, week, marketing_spend, footfall, conversions, revenue FROM store_weeks",
    name="store-weeks",
)
source.to_frame().head()

## Straight to a causal model

A source-backed twin, discovery + training in one pass, and a question:

In [ ]:
twin = ws.create_twin("Store weeks", source_id=source.id)
twin.run_pipeline()

In [ ]:
result = twin.intervene({"marketing_spend": rc.pct(+15)}, outcomes=["revenue"])
result.to_frame()

The loop from here is the same as any other source: extend or re-import on a
schedule, `twin.update()` to fold new rows in, and the [temporal-panel notebook](temporal-panel.ipynb)
for time series and per-environment modelling.